<a href="https://colab.research.google.com/github/rizzken/MedRAG-Eval/blob/main/02_ragas_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ragas
!pip install langchain-google-vertexai -q


  Using cached jiter-0.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
Using cached jiter-0.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (360 kB)
  Attempting uninstall: jiter
    Found existing installation: jiter 0.15.0
    Uninstalling jiter-0.15.0:
      Successfully uninstalled jiter-0.15.0


In [ ]:
# Base packages for this section
!pip install -qU langchain-core langchain-text-splitters langchain-community langchain-chroma langchain-ollama beautifulsoup4 lxml

# If you hit: deepeval requires click<8.4.0 but click 8.4.1 is installed
# run the three commands below to fix and verify dependency conflicts.
!python -m pip install "click>=8.0.0,<8.4.0"
!python -m pip install --upgrade --force-reinstall "deepeval==4.0.4"
!python -m pip check

  Using cached deepeval-4.0.4-py3-none-any.whl.metadata (27 kB)
  Using cached aiohttp-3.13.5-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached grpcio-1.80.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached openai-2.38.0-py3-none-any.whl.metadata (31 kB)
  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached posthog-7.16.2-py3-none-any.whl.metadata (4.7 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pydantic_settings-2.14.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached pyfig

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_ollama import ChatOllama

/tmp/ipykernel_4788/2351560790.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [ ]:
!apt-get install -y zstd
# Install ollama
!curl -fsSL https://ollama.com/install.sh | sh

!pip install ollama

# Start ollama server in the background
!ollama serve &> ollama.log &

import time
time.sleep(10)

print("Ollama installed and server started")

!ollama pull qwen2.5:7b
!ollama pull nomic-embed-text:latest


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama installed and server started




In [ ]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:7b",
    temperature=0,
    max_tokens = 200
)

In [20]:
# Load data from Web
loader = WebBaseLoader(["https://www.webmd.com/cancer/what-is-chordoma",
                        "https://www.webmd.com/brain/spinal-muscular-atrophy",
                        "https://www.webmd.com/back-pain/spinal-stenosis",
                        "https://www.webmd.com/pain-management/pain-management-treatment-overview"])
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever(
    search_kwargs={"k": 10}
)

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}

    Be accurate and specific. If the answer is not in the context, say you don't know.

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()


KeyboardInterrupt: 

In [ ]:
import sys
import types

vertexai_chat_module = types.ModuleType("langchain_community.chat_models.vertexai")
vertexai_chat_module.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = vertexai_chat_module

vertexai_llm_module = types.ModuleType("langchain_community.llms.vertexai")
vertexai_llm_module.VertexAI = type("VertexAI", (), {})
sys.modules["langchain_community.llms.vertexai"] = vertexai_llm_module

In [21]:
from ragas.metrics import LLMContextRecall, NoiseSensitivity
from ragas.llms import LangchainLLMWrapper
from ragas import (EvaluationDataset, evaluate)

test_cases = [
    {"question": "What are the types of SMA?",
     "ground_truth": "SMA has 4 types..."},
    {"question": "What causes spinal stenosis?",
     "ground_truth": "Spinal stenosis is caused by..."},
]

data = [
    {
        "question": [],
        "answer": [],
        "contexts": [],
        "ground_truth": []
        }
]

for tc in test_cases:
    docs = retriever.invoke(tc["question"])
    data["question"].append(tc["question"])
    data["contexts"].append([doc.page_content for doc in docs])
    data["answer"].append(chain.invoke(tc["question"]))
    data["ground_truth"].append(tc["ground_truth"])

evaluator_llm = LangchainLLMWrapper(llm)

evaluation_dataset = EvaluationDataset.from_list(data)

result = evaluate(dataset=evaluation_dataset,
                  metrics=[LLMContextRecall(),
                           NoiseSensitivity()],
                  llm = evaluator_llm)


/tmp/ipykernel_4788/516435551.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, NoiseSensitivity
/tmp/ipykernel_4788/516435551.py:1: DeprecationWarning: Importing NoiseSensitivity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NoiseSensitivity
  from ragas.metrics import LLMContextRecall, NoiseSensitivity


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [ ]:
result.to_pandas()